# List Columns

Suppose your database stores contractors, and each contractor has a variable-length list of skills. If your database supports array-valued columns, these are very likely stored in such a column rather than in a separate table with a foreign key to contractors.

Now suppose you want to filter for contractors who have two specific skills — will your semantic layer allow that? How about counting the contractors that have each particular skill? Or showing the complete list of skills available from contractors in each city?

SLayer types such a column as `ARRAY` with an `element_type` (recognizing these automatically when ingesting schemas), and gives it semantics of its own:

- the **list functions** `contains`, `contains_any`, `contains_all` and `length` test a whole list;
- the list's **elements** are a child dataset `<column>.value` you can filter, group and aggregate, without fanning out the parent;
- `collect(x)` goes the other way and aggregates values *into* a list.

This notebook uses the two list columns of the Jaffle Shop demo: `products.ingredients` (the supplies a product is made from) and `orders.item_skus` (the SKU of each unit on an order, repeats kept).

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_colwidth = 120
pd.options.display.width = 250

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

from slayer.core.errors import SlayerError
from slayer.core.models import column_type_text

engine, storage, models = ensure_jaffle_shop()


def run(query, **kwargs):
    return pd.DataFrame(engine.execute_sync(query, **kwargs).data)


def refused(query):
    try:
        engine.execute_sync(query)
    except SlayerError as error:
        print(error)

## The list type

Ingestion reads each database's native list type (`VARCHAR[]` here; BigQuery `REPEATED`, ClickHouse `Array(T)`, Snowflake `ARRAY`, …) as `type: ARRAY` with the element type it holds. The column's type text tells an agent what a list takes:

In [2]:
by_name = {m.name: m for m in models}
for model_name, column_name in [("products", "ingredients"), ("orders", "item_skus")]:
    column = by_name[model_name].get_column(column_name)
    print(f"{model_name}.{column_name}: {column_type_text(column)}")

products.ingredients: ARRAY<TEXT> (list: contains(ingredients, x), contains_any / contains_all(ingredients, [...]), length(ingredients); its elements are ingredients.value)
orders.item_skus: ARRAY<TEXT> (list: contains(item_skus, x), contains_any / contains_all(item_skus, [...]), length(item_skus); its elements are item_skus.value)


## Testing a whole list

`contains(list, x)` asks whether `x` is one of the list's elements; `contains_all` / `contains_any` compare it with another list, here an array literal; `length` counts the elements. They work wherever an expression does — filters, computed dimensions, measures.

In [3]:
run({
    "source_model": "products",
    "dimensions": ["name", {"expression": "length(ingredients)", "name": "ingredient_count"}],
    "filters": ["contains(ingredients, 'bread')"],
})

,products.name,products.ingredient_count
0,doctor stew,7
1,flame impala,9
2,mel-bun,8
3,nutellaphone who dis?,7
4,the krautback,9


In [4]:
run({
    "source_model": "products",
    "dimensions": ["name"],
    "filters": ["contains_all(ingredients, ['bread', 'cheese'])"],
})

,products.name
0,doctor stew
1,flame impala
2,mel-bun
3,the krautback


In [5]:
run({
    "source_model": "products",
    "dimensions": ["name"],
    "filters": ["contains_any(ingredients, ['banana', 'chai mix'])"],
})

,products.name
0,chai and mighty
1,nutellaphone who dis?


Absence is a `NOT contains(...)`: the products made without bread.

In [6]:
run({
    "source_model": "products",
    "dimensions": ["type"],
    "measures": ["count(*)"],
    "filters": ["NOT contains(ingredients, 'bread')"],
})

,products.type,products._count
0,beverage,5


A list-valued `{variable}` in a list argument binds as a list, so a "basket" can come from the caller:

In [7]:
run(
    {"source_model": "orders", "measures": ["count(*)"], "filters": ["contains_any(item_skus, {wanted})"]},
    variables={"wanted": ["JAF-004", "JAF-005"]},
)

,orders._count
0,74520


## A list is not a scalar

A list has no single value to group by, compare with `=`, or sum, so those are refused with a typed error that names what to do instead. `contains` reads only lists; on text, `LIKE` does the substring test.

In [8]:
refused({"source_model": "products", "dimensions": ["ingredients"]})

ListTypeError: Dimension 'ingredients' is a list, which cannot be grouped on.
  suggestion: Group by its elements: ingredients.value.


In [9]:
refused({"source_model": "products", "dimensions": ["name"], "filters": ["ingredients = 'bread'"]})

ListTypeError: `ingredients` is a list and cannot be compared with ==.
  suggestion: Test membership with contains(ingredients, x), or compare its elements through ingredients.value.


In [10]:
refused({"source_model": "orders", "measures": ["sum(item_skus)"]})

ListTypeError: `item_skus` is a list: an aggregation over a list is only count; sum() cannot take it.
  suggestion: Aggregate its elements: sum(item_skus.value).


In [11]:
refused({"source_model": "products", "dimensions": ["name"], "filters": ["contains(name, 'stew')"]})

ListTypeError: contains() reads a list; `name` is not a list.
  suggestion: Use contains(list, value) for one value; on text, match a substring with name LIKE '%…%'.


## Elements as rows

Every list column is also a one-to-many hop to its elements: `ingredients.value` is one row per non-NULL element, and `count(ingredients.*)` counts those rows. Group by the element, and `to_many_handling: "associate"` makes `count(*)` the number of **products** using each ingredient:

The warning is association's usual reminder: a product counts once under each of its ingredients, so the cells overlap and do not add up to the product total.

In [12]:
run({
    "source_model": "products",
    "dimensions": ["ingredients.value"],
    "measures": ["count(*)"],
    "to_many_handling": "associate",
})

,products.ingredients.value,products._count
0,16oz compostable clear cup,5
1,16oz compostable clear lid,5
2,banana,1
3,beef stew,1
4,biodegradable straw,5
5,bread,5
6,chai mix,1
7,cheese,4
8,coffee,2
9,compostable cutlery - knife,5


On orders, `count(item_skus.*)` counts units while the associated `count(orders.*)` counts orders, so the two differ by the orders holding a SKU more than once:

In [13]:
run({
    "source_model": "orders",
    "dimensions": ["item_skus.value"],
    "measures": [
        {"formula": "count(orders.*)", "name": "orders_with_sku"},
        {"formula": "count(item_skus.*)", "name": "units"},
    ],
    "to_many_handling": "associate",
})

,orders.item_skus.value,orders.orders_with_sku,orders.units
0,BEV-001,144357,149873
1,BEV-002,143870,149380
2,BEV-003,144504,149976
3,BEV-004,143745,149302
4,BEV-005,143671,149186
5,JAF-001,40780,44818
6,JAF-002,40447,44261
7,JAF-003,40453,44350
8,JAF-004,40572,44570
9,JAF-005,40359,44417


## No fan-out

Reading the elements never multiplies the parent rows: each aggregation runs over its own rows, so an order measure next to an element measure keeps its value. The order counts and revenue below are the same with or without the `units` column beside them.

In [14]:
with_units = run({
    "source_model": "orders",
    "dimensions": ["stores.name"],
    "measures": ["count(*)", "sum(order_total)", {"formula": "count(item_skus.*)", "name": "units"}],
})
without_units = run({
    "source_model": "orders",
    "dimensions": ["stores.name"],
    "measures": ["count(*)", "sum(order_total)"],
})
with_units

,orders.stores.name,orders._count,orders.order_total_sum,orders.units
0,Brooklyn,255512,2745528.62,374833
1,Chicago,104612,1169578.29,155207
2,New Orleans,15942,179638.67,24177
3,Philadelphia,192817,2089087.21,279950
4,San Francisco,88606,1044581.66,135966


In [15]:
with_units.drop(columns="orders.units").equals(without_units)

True

## Every element condition reads the same element

All `item_skus.value` conditions in one query constrain **one** element, so asking for a jaffle and a drink with two element filters matches nothing — no single SKU is both. To require several elements, use `contains_all`:

In [16]:
run({
    "source_model": "orders",
    "measures": ["count(*)"],
    "filters": ["item_skus.value = 'JAF-001'", "item_skus.value = 'BEV-001'"],
})

,orders._count
0,0


In [17]:
run({
    "source_model": "orders",
    "measures": ["count(*)"],
    "filters": ["contains_all(item_skus, ['JAF-001', 'BEV-001'])"],
})

,orders._count
0,11605


## Collecting values into a list

`collect(x)` is the aggregation that builds a list: the distinct non-NULL values of `x` in each cell, sorted. It takes columns, elements and expressions alike:

In [18]:
run({
    "source_model": "products",
    "dimensions": ["type"],
    "measures": [{"formula": "collect(ingredients.value)", "name": "all_ingredients"}],
})

,products.type,products.all_ingredients
0,beverage,"[16oz compostable clear cup, 16oz compostable clear lid, biodegradable straw, chai mix, coffee, french vanilla syrup..."
1,jaffle,"[banana, beef stew, bread, cheese, compostable cutlery - knife, cutlery - fork, ghost pepper sauce, house-pickled ca..."


`collect` works on regular columns too — here, the products of each type:

In [19]:
run({
    "source_model": "products",
    "dimensions": ["type"],
    "measures": [{"formula": "collect(name)", "name": "product_names"}],
})

,products.type,products.product_names
0,beverage,"[adele-ade, chai and mighty, for richer or pourover , tangaroo, vanilla ice]"
1,jaffle,"[doctor stew, flame impala, mel-bun, nutellaphone who dis?, the krautback]"


A cell with no values gets the empty list, never `NULL` — here the orders that have no items:

In [20]:
run({
    "source_model": "orders",
    "dimensions": [{"expression": "length(item_skus)", "name": "item_count"}],
    "measures": ["count(*)", {"formula": "collect(item_skus.value)", "name": "skus"}],
    "filters": ["length(item_skus) < 2"],
})

,orders.item_count,orders._count,orders.skus
0,0,4522,[]
1,1,469224,"[BEV-001, BEV-002, BEV-003, BEV-004, BEV-005, JAF-001, JAF-002, JAF-003, JAF-004, JAF-005]"


## Lists across stages

A collected list stays a list in a later stage, with the same functions and elements. The first stage builds each customer's basket of SKUs ever ordered; the second counts the customers whose basket holds `JAF-005`, and how many customers ever bought each SKU:

In [21]:
baskets = {
    "name": "baskets",
    "source_model": "orders",
    "dimensions": ["customer_id"],
    "measures": [{"formula": "collect(item_skus.value)", "name": "skus"}],
}
run([baskets, {"source_model": "baskets", "measures": ["count(*)"], "filters": ["contains(skus, 'JAF-005')"]}])

,baskets._count
0,1663


In [22]:
run([
    baskets,
    {"source_model": "baskets", "dimensions": ["skus.value"], "measures": ["count(*)"], "to_many_handling": "associate"},
])

,baskets.skus.value,baskets._count
0,BEV-001,2579
1,BEV-002,2577
2,BEV-003,2575
3,BEV-004,2574
4,BEV-005,2575
5,JAF-001,1657
6,JAF-002,1659
7,JAF-003,1663
8,JAF-004,1668
9,JAF-005,1663


## Where lists run

Lists, the list functions and `collect` run on Postgres, DuckDB, BigQuery, Snowflake, ClickHouse, Trino and Presto; on a database without native lists (SQLite, MySQL, SQL Server, …) every list construct fails at plan time. See [List columns](../../concepts/queries.md#list-columns) and [Lists](../../concepts/references.md#lists) for the full rules.